In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
FWD_MS_COL = "forward_ms"
BWD_MS_COL = "backward_ms"
FWD_MEM_COL = "forward_memory_mb"
BWD_MEM_COL = "backward_memory_mb"
MS_TOTAL = "ms_total"

CONV_COL = "conv_type"
DATASET_COL = "dataset"
BACKEND_COL = "backend"
HEADS_COL = "heads"
FEATURE_DIM_COL = "feature_dim"

In [3]:
graph_transformer_out = "/home/fvelikon/projects/cuda_exp/out/gt_no_reordering"
graph_transformer_out = "/home/fvelikon/projects/cuda_exp/out/gt_new"
gatv2_out = "/home/fvelikon/projects/cuda_exp/out/gatv2_no_reordering"
gatv2_out = "/home/fvelikon/projects/cuda_exp/out/test_graph_reordering"
min_aggr_out = "/home/fvelikon/projects/cuda_exp/out/min_aggr_no_reordering"

In [4]:
REORDERING_COL = "graph_reordering_partition_size"

# other columns used for tuning parameters

In [5]:
BASELINE = "dgl"

def add_speedup_columns(df_in: pd.DataFrame, baseline: str = BASELINE) -> pd.DataFrame:
    """
    For each (dataset, feature_dim, heads) triple, compute speedup
    vs `baseline` separately for forward_ms and backward_ms.
    """
    df = df_in.copy()
    idx_cols = [CONV_COL, DATASET_COL, FEATURE_DIM_COL, HEADS_COL, REORDERING_COL]

    for metric in [FWD_MS_COL, BWD_MS_COL, FWD_MEM_COL, BWD_MEM_COL, MS_TOTAL]:
        base_col = f"{metric}_baseline_{baseline}"
        speed_col = f"{metric}_speedup_vs_{baseline}"
        df[base_col] = np.nan
        df[speed_col] = np.nan

        for key, group in df.groupby(idx_cols):
            base_rows = group[group["backend"] == baseline]
            if base_rows.empty:
                continue
            base_val = base_rows.iloc[0][metric]
            if not np.isfinite(base_val) or base_val <= 0:
                continue

            mask = (
                (df[CONV_COL] == key[0]) &
                (df[DATASET_COL] == key[1]) &
                (df[FEATURE_DIM_COL] == key[2]) &
                (df[HEADS_COL] == key[3]) &
                (df[REORDERING_COL] == key[4])
            )
            df.loc[mask, base_col] = base_val
            df.loc[mask, speed_col] = base_val / df.loc[mask, metric]

    return df


In [6]:
gcn_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_no_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_torch_native", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_cusoarse_with_precompute_bwd", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_partition", index_col=0),
])
gcn_df[HEADS_COL] = 1


# MIN AGGR:
min_aggr_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_no_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_partition", index_col=0),
])
min_aggr_df[HEADS_COL] = 1

#gatv2

gatv2_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/test_graph_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gatv2_partition", index_col=0),
])

# graph transformer
gt_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_new", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_partition", index_col=0),
])


df = pd.concat([
    gcn_df,
    min_aggr_df,
    gatv2_df,
    gt_df,
])

In [7]:
df[MS_TOTAL] = df[FWD_MS_COL] + df[BWD_MS_COL]
df = add_speedup_columns(df, BASELINE)
df["density"] = df["num_edges"] / df["num_nodes"] / (df["num_nodes"] - 1)
df["average_node_degree"] = df["num_edges"] / df["num_nodes"]


In [23]:
def build_speedup_table_with_dataset_index(
    df_speed: pd.DataFrame,
    baseline: str,
) -> pd.DataFrame:
    """
    Build a single speedup table with a MultiIndex including dataset.

    Index:
        ('conv_type', 'dataset', 'heads', 'feature_dim')  if 'heads' in df_speed.columns
        ('conv_type', 'dataset', 'feature_dim')          otherwise

    Columns:
        MultiIndex (metric_short, backend), where metric_short ∈
        {'Fwd time', 'Bwd time', 'Fwd mem', 'Bwd mem'} and values are
        speedup vs baseline (>1 = better than baseline).

    Baseline backend column is dropped (speedup is always 1.0).
    """
    metrics = [
        (FWD_MS_COL,  "Fwd time"),
        (BWD_MS_COL,  "Bwd time"),
        (FWD_MEM_COL, "Fwd mem"),
        (BWD_MEM_COL, "Bwd mem"),
        (MS_TOTAL, "Fwd + Bwd time")
    ]

    # decide index cols
    idx_cols = [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL]

    metric_frames = []

    for metric, short_name in metrics:
        speed_col = f"{metric}_speedup_vs_{baseline}"
        if speed_col not in df_speed.columns:
            continue

        df_metric = df_speed[np.isfinite(df_speed[speed_col])].copy()
        if df_metric.empty:
            continue

        pivot = df_metric.pivot_table(
            index=idx_cols,
            columns=BACKEND_COL,
            values=speed_col,
            aggfunc="max",
        )

        # drop baseline backend (always 1.0)
        pivot = pivot.drop(columns=[baseline], errors="ignore")

        if pivot.empty:
            continue

        # attach metric name as top-level column
        pivot.columns = pd.MultiIndex.from_product(
            [[short_name], pivot.columns],
            names=["metric", "backend"],
        )

        metric_frames.append(pivot)

    if not metric_frames:
        return pd.DataFrame()

    table = pd.concat(metric_frames, axis=1)

    # sort rows & columns
    table = table.sort_index()
    table = table.sort_index(axis=1)

    return table
def build_raw_table_with_dataset_index(
    df: pd.DataFrame,
    index = None,
) -> pd.DataFrame:
    """
    Build a raw-metrics table with the same structure as
    `build_speedup_table_with_dataset_index`, but containing
    *raw* runtimes (ms) and memory (MB) for all backends.

    Index:
        (conv_type, dataset, heads, feature_dim)

    Columns:
        MultiIndex (metric_short, backend), where metric_short ∈
        {'Fwd time', 'Bwd time', 'Fwd mem', 'Bwd mem', 'Fwd + Bwd time'}.

    Values:
        Raw numbers from the corresponding metric columns.
    """
    metrics = [
        (FWD_MS_COL,  "Fwd time"),
        (BWD_MS_COL,  "Bwd time"),
        (FWD_MEM_COL, "Fwd mem"),
        (BWD_MEM_COL, "Bwd mem"),
        (MS_TOTAL,    "Fwd + Bwd time"),
    ]

    idx_cols = index or [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL]

    metric_frames: list[pd.DataFrame] = []

    for metric, short_name in metrics:
        if metric not in df.columns:
            continue

        df_metric = df[np.isfinite(df[metric])].copy()
        if df_metric.empty:
            continue

        pivot = df_metric.pivot_table(
            index=idx_cols,
            columns=BACKEND_COL,
            values=metric,
            aggfunc="min",  # same choice as in speedup table
        )

        if pivot.empty:
            continue

        # put metric name on top level
        pivot.columns = pd.MultiIndex.from_product(
            [[short_name], pivot.columns],
            names=["metric", "backend"],
        )

        metric_frames.append(pivot)

    if not metric_frames:
        return pd.DataFrame()

    table = pd.concat(metric_frames, axis=1)

    table = table.sort_index()
    table = table.sort_index(axis=1)

    return table


In [16]:
no_reordering_df = df[df[REORDERING_COL] == -1]
[
    [
        CONV_COL,
        FWD_MS_COL,
        BWD_MS_COL,
        FWD_MEM_COL,
        BWD_MEM_COL,
        DATASET_COL,
        BACKEND_COL,
        HEADS_COL,
        FEATURE_DIM_COL,
        REORDERING_COL,
    ]
]
reordering_df = df[df[REORDERING_COL] != -1]

[
    [
        CONV_COL,
        FWD_MS_COL,
        BWD_MS_COL,
        FWD_MEM_COL,
        BWD_MEM_COL,
        DATASET_COL,
        BACKEND_COL,
        HEADS_COL,
        FEATURE_DIM_COL,
        REORDERING_COL,
    ]
]

[['conv_type',
  'forward_ms',
  'backward_ms',
  'forward_memory_mb',
  'backward_memory_mb',
  'dataset',
  'backend',
  'heads',
  'feature_dim',
  'graph_reordering_partition_size']]

In [17]:
datasets_df = df.groupby(DATASET_COL)[["num_nodes", "num_edges"]].max().reset_index()
datasets_df["density"] = datasets_df["num_edges"] / datasets_df["num_nodes"] / (datasets_df["num_nodes"] - 1)
datasets_df["average_node_degree"] = datasets_df["num_edges"] / datasets_df["num_nodes"]

datasets_df = datasets_df.sort_values(by="density", ascending=False).reset_index(drop=True)

In [18]:
print(datasets_df.to_markdown())

|    | dataset       |   num_nodes |   num_edges |     density |   average_node_degree |
|---:|:--------------|------------:|------------:|------------:|----------------------:|
|  0 | hm-prices     |       46563 |    21461990 | 0.00989914  |             460.924   |
|  1 | hm-categories |       46563 |    21461990 | 0.00989914  |             460.924   |
|  2 | tolokers-2    |       11758 |     1038000 | 0.00750875  |              88.2803  |
|  3 | avazu-ctr     |       76269 |    21968154 | 0.00377662  |             288.035   |
|  4 | cora          |        2708 |       10556 | 0.00144     |               3.89808 |
|  5 | citeseer      |        3327 |        9104 | 0.00082273  |               2.7364  |
|  6 | twitch-views  |      168114 |    13595114 | 0.000481036 |              80.8684  |
|  7 | pubmed        |       19717 |       88648 | 0.000228039 |               4.49602 |
|  8 | artnet-views  |       50405 |      560696 | 0.000220693 |              11.1238  |
|  9 | artnet-exp    

In [19]:
datasets_df["dataset"].values.tolist()

['hm-prices',
 'hm-categories',
 'tolokers-2',
 'avazu-ctr',
 'cora',
 'citeseer',
 'twitch-views',
 'pubmed',
 'artnet-views',
 'artnet-exp',
 'city-reviews',
 'city-roads-M',
 'ogbn-arxiv',
 'ogbn-products',
 'city-roads-L',
 'pokec-regions',
 'web-fraud',
 'web-topics',
 'web-traffic']

## GT

In [20]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gt"], "dgl").to_markdown(floatfmt=".2f"))

|                                 |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cuda') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cuda') |   ('Fwd time', 'triton_block_sparse') |
|:--------------------------------|----------------------:|-------------------------------------:|-----------------------:|--------------------------------------:|-----------------------------:|--------------------------------------------:|----------------------:|-------------------------------------:|-----------------------:|--------------------------------------:|
| ('gt', 'artnet-exp', 2, 256)    |                  1.04 |                                 1.19 |                   1.11 |                                  0.63 |                         1.11 |                                        0.77 |                  1.21

In [21]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gt"]).to_markdown(floatfmt=".2f"))

|                                 |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'dgl') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cuda') |   ('Bwd time', 'dgl') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'dgl') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cuda') |   ('Fwd time', 'dgl') |   ('Fwd time', 'triton_block_sparse') |
|:--------------------------------|----------------------:|---------------------:|-------------------------------------:|-----------------------:|----------------------:|--------------------------------------:|-----------------------------:|----------------------------:|--------------------------------------------:|----------------------:|---------------------:|-------------------------------------:|-----------------------:|----------------------:|--------------------------------------:|
| ('gt', 'artn

In [24]:
print(build_raw_table_with_dataset_index(df[df[CONV_COL] == "gt"], [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL, REORDERING_COL]).to_markdown(floatfmt=".2f"))

|                                       |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'dgl') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cuda') |   ('Bwd time', 'dgl') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'dgl') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cuda') |   ('Fwd time', 'dgl') |   ('Fwd time', 'triton_block_sparse') |
|:--------------------------------------|----------------------:|---------------------:|-------------------------------------:|-----------------------:|----------------------:|--------------------------------------:|-----------------------------:|----------------------------:|--------------------------------------------:|----------------------:|---------------------:|-------------------------------------:|-----------------------:|----------------------:|--------------------------------------:|
| 

## GATv2

In [31]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gat_v2"], "dgl").to_markdown(floatfmt=".2f"))

|                                    |   ('Bwd mem', 'cuda') |   ('Bwd time', 'cuda') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd mem', 'cuda') |   ('Fwd time', 'cuda') |
|:-----------------------------------|----------------------:|-----------------------:|-----------------------------:|----------------------:|-----------------------:|
| ('gat_v2', 'artnet-exp', 2, 256)   |                  5.47 |                   1.60 |                         1.85 |                  8.42 |                   2.45 |
| ('gat_v2', 'artnet-exp', 2, 512)   |                  5.47 |                   1.39 |                         1.54 |                  8.58 |                   1.96 |
| ('gat_v2', 'artnet-exp', 4, 256)   |                  6.03 |                   1.54 |                         1.82 |                  9.70 |                   2.53 |
| ('gat_v2', 'artnet-exp', 4, 512)   |                  6.02 |                   1.50 |                         1.64 |                  9.87 |                  

In [32]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gat_v2"]).to_markdown(floatfmt=".2f"))

|                                     |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'dgl') |   ('Bwd time', 'cuda') |   ('Bwd time', 'dgl') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'dgl') |   ('Fwd time', 'cuda') |   ('Fwd time', 'dgl') |
|:------------------------------------|----------------------:|---------------------:|-----------------------:|----------------------:|-----------------------------:|----------------------------:|----------------------:|---------------------:|-----------------------:|----------------------:|
| ('gat_v2', 'artnet-exp', 2, 256)    |                929.10 |              5083.21 |                   8.95 |                 14.36 |                        12.59 |                       23.25 |                426.46 |              3589.37 |                   3.63 |                  8.89 |
| ('gat_v2', 'artnet-exp', 2, 512)    |               1822.22 |              9960.55 |                  26.78 |          

## GCN

In [16]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gcn"], "dgl").to_markdown(floatfmt=".2f"))

|                                  |   ('Bwd mem', 'cusparse') |   ('Bwd mem', 'cusparse_precomputed_bwd') |   ('Bwd mem', 'fusegnn') |   ('Bwd mem', 'pyg') |   ('Bwd mem', 'tcgnn') |   ('Bwd mem', 'torch_native_gcn') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cusparse') |   ('Bwd time', 'cusparse_precomputed_bwd') |   ('Bwd time', 'fusegnn') |   ('Bwd time', 'pyg') |   ('Bwd time', 'tcgnn') |   ('Bwd time', 'torch_native_gcn') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cusparse') |   ('Fwd + Bwd time', 'cusparse_precomputed_bwd') |   ('Fwd + Bwd time', 'fusegnn') |   ('Fwd + Bwd time', 'pyg') |   ('Fwd + Bwd time', 'tcgnn') |   ('Fwd + Bwd time', 'torch_native_gcn') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cusparse') |   ('Fwd mem', 'cusparse_precomputed_bwd') |   ('Fwd mem', 'fusegnn') |   ('Fwd mem', 'pyg') |   ('Fwd mem', 'tcgnn') |   ('Fwd mem', 'torch_native_gcn') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cus

In [28]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gcn"]).to_markdown(floatfmt=".2f"))

|                                  |   ('Bwd mem', 'cusparse') |   ('Bwd mem', 'cusparse_precomputed_bwd') |   ('Bwd mem', 'dgl') |   ('Bwd mem', 'fusegnn') |   ('Bwd mem', 'pyg') |   ('Bwd mem', 'tcgnn') |   ('Bwd mem', 'torch_native_gcn') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cusparse') |   ('Bwd time', 'cusparse_precomputed_bwd') |   ('Bwd time', 'dgl') |   ('Bwd time', 'fusegnn') |   ('Bwd time', 'pyg') |   ('Bwd time', 'tcgnn') |   ('Bwd time', 'torch_native_gcn') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cusparse') |   ('Fwd + Bwd time', 'cusparse_precomputed_bwd') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd + Bwd time', 'fusegnn') |   ('Fwd + Bwd time', 'pyg') |   ('Fwd + Bwd time', 'tcgnn') |   ('Fwd + Bwd time', 'torch_native_gcn') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cusparse') |   ('Fwd mem', 'cusparse_precomputed_bwd') |   ('Fwd mem', 'dgl') |   ('Fwd mem', 'fusegnn') |   ('Fwd mem', 'pyg') |   ('Fwd mem', 'tcgn

## MinAggr

In [33]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "min_aggr"], "dgl").to_markdown(floatfmt=".2f"))

|                                       |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'cugraph') |   ('Bwd time', 'cuda') |   ('Bwd time', 'cugraph') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'cugraph') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'cugraph') |   ('Fwd time', 'cuda') |   ('Fwd time', 'cugraph') |
|:--------------------------------------|----------------------:|-------------------------:|-----------------------:|--------------------------:|-----------------------------:|--------------------------------:|----------------------:|-------------------------:|-----------------------:|--------------------------:|
| ('min_aggr', 'artnet-exp', 1, 64)     |                  1.41 |                     1.11 |                   1.26 |                      0.79 |                         1.67 |                            0.89 |                  1.67 |                     1.16 |                   2.26 |                      0.96 |
| ('min_aggr', 'artnet-exp', 1, 128)    |              

In [34]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "min_aggr"]).to_markdown(floatfmt=".2f"))

|                                       |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'cugraph') |   ('Bwd mem', 'dgl') |   ('Bwd time', 'cuda') |   ('Bwd time', 'cugraph') |   ('Bwd time', 'dgl') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'cugraph') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'cugraph') |   ('Fwd mem', 'dgl') |   ('Fwd time', 'cuda') |   ('Fwd time', 'cugraph') |   ('Fwd time', 'dgl') |
|:--------------------------------------|----------------------:|-------------------------:|---------------------:|-----------------------:|--------------------------:|----------------------:|-----------------------------:|--------------------------------:|----------------------------:|----------------------:|-------------------------:|---------------------:|-----------------------:|--------------------------:|----------------------:|
| ('min_aggr', 'artnet-exp', 1, 64)     |                 94.94 |                   120.73 |               133.97 |       

In [1]:
import pandas as pd


In [4]:
gt_kernel_ncu = pd.read_csv("../gt_profile.csv")
gt_kernel_ncu

,ID,Process ID,Process Name,Host Name,Kernel Name,Context,Stream,Block Size,Grid Size,Device,...,smsp__warps_active.min.peak_sustained,smsp__warps_active.min.per_cycle_active,smsp__warps_active.sum.peak_sustained,smsp__warps_active.sum.per_cycle_active,smsp__warps_eligible.avg.per_cycle_active,smsp__warps_eligible.max.per_cycle_active,smsp__warps_eligible.min.per_cycle_active,smsp__warps_eligible.sum.per_cycle_active,thread_inst_executed,thread_inst_executed_true
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,warp,warp,warp,warp,warp,warp,warp,warp,inst,inst
1,0.0,2235973.0,python3.11,127.0.0.1,"GraphAttentionForward_CSR_MH(unsigned long, un...",1.0,7.0,"(32, 1, 1)","(500, 1, 1)",0.0,...,16,0.66,6912,495.97,0.09,0.24,0.04,40.20,26531000,25394500
2,1.0,2235973.0,python3.11,127.0.0.1,"GraphAttentionForward_CSR_MH(unsigned long, un...",1.0,7.0,"(32, 1, 1)","(500, 1, 1)",0.0,...,16,0.65,6912,496.06,0.09,0.24,0.04,40.10,26531000,25394500
